# **Fixed, Dynamic and Ablation Analysis**

This file has the following part: 

$\S0$ **Setup**: Define global constants, and load packages \
$\S1$ **Reward vs Steps**: Plot cumulative rewards against search steps to compare the performance of the heuristics \
$\S2$ **Search**: Visualise the exploration patterns of the fixed and dynamic heuristics in weight space \
$\S3$ **Distance**: Plot histograms of nearest-neighbour distances to compare the exploratory behaviour of the fixed, dynamic and ablation searches

## $\S0$ Setup

In [ ]:
#Constants
dim_X = 4 #dimension of hypersurface
alpha_X = -1 #adjunction number of hypersurface

In [ ]:
#Packages

#Storage
import os
import pandas as pd
import json

#Distances
import numpy as np
from scipy.spatial import cKDTree

#Figures
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, MaxNLocator

Load Data

In [ ]:
#Fixed Heuristic Search Data
fixed_settings = json.load(open("../data/fixed/fixed_approximation/fixed_approximation_settings.json"))
fixed_terminal_points = pd.read_csv(f"../data/fixed/fixed_certification/fixed_certification_points.csv", header=None).values.astype(int)
fixed_searched_points = pd.read_csv(f"../data/fixed/fixed_searched_points/fixed_searched_points/fixed_searched_points.csv", header=None).values.astype(int)
fixed_reward_vs_steps = pd.read_csv(f"../data/fixed/fixed_reward_vs_steps/fixed_reward_vs_steps.csv", header=None).values.astype(int)

#Dynamic Heuristic Search Data
dynamic_settings = json.load(open("../data/dynamic/dynamic_approximation/dynamic_approximation_settings.json"))
max_episodes = dynamic_settings["max_episodes"]
state_dim = dynamic_settings["state_dim"]
dynamic_terminal_points_total = pd.read_csv(f"../data/dynamic/dynamic_certification/dynamic_certification_points_total.csv", header=None).values.astype(int)
dynamic_terminal_points_episodal = []
dynamic_searched_points_episodal = []
dynamic_reward_vs_steps_episodal = []
for i in range(1, max_episodes + 1):
    dynamic_terminal_points_episodal.append(pd.read_csv(f"../data/dynamic/dynamic_certification/dynamic_certification_points_episode_{i}.csv", header=None).values.astype(int))
    dynamic_searched_points_episodal.append(pd.read_csv(f"../data/dynamic/dynamic_searched_points/dynamic_searched_points_episode_{i}.csv", header=None).values.astype(int))
    dynamic_reward_vs_steps_episodal.append(pd.read_csv(f"../data/dynamic/dynamic_reward_vs_steps/dynamic_reward_vs_steps_episode_{i}.csv", header=None).values.astype(int))

#Ablation Heuristic Search Data
ablation_settings = json.load(open("../data/ablation/ablation_approximation/ablation_approximation_settings.json"))
max_episodes = dynamic_settings["max_episodes"]
ablation_terminal_points_total = pd.read_csv(f"../data/ablation/ablation_certification/ablation_certification_points_total.csv", header=None).values.astype(int)
ablation_terminal_points_episodal = []
ablation_searched_points_episodal = []
ablation_reward_vs_steps_episodal = []
for i in range(1, max_episodes + 1):
    ablation_terminal_points_episodal.append(pd.read_csv(f"../data/ablation/ablation_certification/ablation_certification_points_episode_{i}.csv", header=None).values.astype(int))
    ablation_searched_points_episodal.append(pd.read_csv(f"../data/ablation/ablation_searched_points/ablation_searched_points_episode_{i}.csv", header=None).values.astype(int))
    ablation_reward_vs_steps_episodal.append(pd.read_csv(f"../data/ablation/ablation_reward_vs_steps/ablation_reward_vs_steps_episode_{i}.csv", header=None).values.astype(int))


## $\S1$  Reward vs Steps

Fixed Dynamic Ablation Reward vs Steps

In [ ]:
# Font settings
plt.rcParams.update({
    "text.usetex": False,       
    "font.family": "serif",     
    "font.serif": ["DejaVu Serif"],  
    "mathtext.fontset": "cm",  
})

# Instantiate plot
fig, ax = plt.subplots(figsize=(7, 6))

#Fixed
fixed_rewards = fixed_reward_vs_steps[:, 1]
fixed_steps = fixed_reward_vs_steps[:, 0]
ax.plot(fixed_steps, fixed_rewards, color='blue', alpha=0.5, linestyle='--', label='Fixed')


#Dynamic
dynamic_rewards_all = np.array([dynamic_reward_vs_steps_episodal[i][:, 1] 
                        for i in range(max_episodes)])
steps = dynamic_reward_vs_steps_episodal[0][:, 0]  
mean_rewards = np.mean(dynamic_rewards_all, axis=0)
std_rewards = np.std(dynamic_rewards_all, axis=0, ddof=1)
# Plot individual runs
for i in range(max_episodes):
    ax.plot(steps, dynamic_rewards_all[i], color='blue', alpha=0.5, linewidth=1.5)
# Plot mean and shaded std
ax.plot(steps, mean_rewards, color='blue', linewidth=3, label='Dyn. Mean')
ax.fill_between(steps, mean_rewards - std_rewards, mean_rewards + std_rewards,
                color='blue', alpha=0.15, linewidth=0, label=r'Dyn. Mean $\pm$ Std')

#Ablation
ablation_rewards_all = np.array([ablation_reward_vs_steps_episodal[i][:, 1] 
                        for i in range(max_episodes)])
ablation_steps = ablation_reward_vs_steps_episodal[0][:, 0]
ablation_mean_rewards = np.mean(ablation_rewards_all, axis=0)
ablation_std_rewards = np.std(ablation_rewards_all, axis=0, ddof=1)
# Plot individual runs
for i in range(max_episodes):
    ax.plot(ablation_steps, ablation_rewards_all[i], color='blue', linestyle='-.', alpha=0.5, linewidth=1.5)
# Plot mean and shaded std
ax.plot(ablation_steps, ablation_mean_rewards, color='blue', linewidth=3, linestyle='-.', label='Abl. Mean')
ax.fill_between(ablation_steps, ablation_mean_rewards - ablation_std_rewards, ablation_mean_rewards + ablation_std_rewards,
                color='blue', alpha=0.15, linewidth=0, hatch='xx', label=r'Abl. Mean $\pm$ Std')

# Format plot
ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}')) 
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax.grid(axis="y", alpha=0.3, linestyle='-')
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines['bottom'].set_color("grey")
ax.tick_params(axis='x', colors="grey")
ax.xaxis.label.set_color("grey")
ax.yaxis.label.set_color("grey")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.yaxis.set_tick_params(width=0)
plt.xticks(rotation=45)
plt.xticks(fontsize=18, color='grey')
plt.yticks(fontsize=18, color='grey')

# Labels and legend
ax.set_xlabel("Step Count", fontsize=20, color='grey', labelpad=15)
ax.set_ylabel("Terminal Points Found", fontsize=20, color='grey', labelpad=15)
plt.legend(frameon=False, fontsize=18, loc='lower right',
            title_fontsize=20)

plt.show()

os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/fixed_dynamic_ablation_rewards_vs_steps.png", dpi=300, bbox_inches="tight")

## $\S2$  Search

Fixed Search

In [ ]:
#Seperate quasismooth and nonquasismooth terminal points for plotting
fixed_terminal_points_qs = []
fixed_terminal_points_nonqs = []
qs = pd.read_csv("../data/quasismooth/quasismooth_terminal.csv", header=None).values.astype(int).tolist()
qs = [tuple(point[1:]) for point in qs]
for point in fixed_terminal_points:
    if tuple(point) in qs:
         fixed_terminal_points_qs.append(tuple(point))
    else:
        fixed_terminal_points_nonqs.append(tuple(point))

#Font settings
plt.rcParams.update({
    "text.usetex": False,       # don't require external LaTeX
    "font.family": "serif",     # generic family (will map automatically)
    "font.serif": ["DejaVu Serif"],  # built-in, always available
    "mathtext.fontset": "cm",   # use Computer Modern for math text
})

#Instantiate plot
fig, axs = plt.subplots(2, 3 , figsize=(30,20))
axs = axs.flatten()

#Project the data via (a1,...,an) -> (ai+1, ai+2) for i=0,...,n-2
for i in range(state_dim - 1):
    ax = axs[i]
    i1 = i
    i2 = i + 1

    #Determine grid size
    max_value = int(max(
        max(point[i1] for point in fixed_searched_points),
        max(point[i2] for point in fixed_searched_points),
    ))

    #Grid points
    X, Y = np.meshgrid(range(1, max_value + 1), range(1, max_value + 1))
    X = X.flatten()
    Y = Y.flatten()

    #Marker size
    ax_width = ax.get_window_extent().width
    ax_height = ax.get_window_extent().height
    marker_size = (min(ax_width/ max_value, ax_height/ max_value) * 0.85) ** 2 

    #Grey points where ai<=aj
    grey_mask = X <= Y
    ax.scatter(X[grey_mask], Y[grey_mask], color='#eeeee4', s=marker_size, alpha=1, label='Searchable')

    #Searched points
    ax_ay_searched = list(set([(point[i1],point[i2]) for point in fixed_searched_points]))
    ax.scatter(*zip(*ax_ay_searched), color='orange', s=marker_size, alpha=1, label='Searched')

    #Quasismooth terminal points
    ax_ay_terminal = list(set([(point[i1],point[i2]) for point in fixed_terminal_points_qs]))
    ax.scatter(*zip(*ax_ay_terminal), color='black', s=marker_size, alpha=1, label='Quasismooth terminal')

    #Nonquasismooth terminal points
    ax_ay_terminal = list(set([(point[i1],point[i2]) for point in fixed_terminal_points_nonqs]))
    ax.scatter(*zip(*ax_ay_terminal), color='blue', s=marker_size, alpha=0.5, label='Nonquasismooth terminal')

    #Lablels and title
    ax.set_xlabel(f'$a_{i1+1}$', fontsize=40, color='grey')
    ax.set_ylabel(f'$a_{i2+1}$', fontsize=40, color='grey')

    #Format plot
    num_bins = 6
    ticks = list(np.linspace(0, max_value, num_bins))
    ticks.append(1)
    ticks = sorted(set(ticks))
    ticks = [int(t) for t in ticks if t != 0]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis='both', which='both', length=0, labelsize=20, colors="grey")

axs[state_dim - 1].axis('off')

#Legend
h, l = ax.get_legend_handles_labels()
axs[state_dim - 1].legend(
    h, l,
    frameon=False,
    markerscale=100,
    loc='center',
    fontsize=40,
    title=r'$\mathbf{Point\ Types}$',
    title_fontsize=45
)

plt.tight_layout()
plt.show()

#Create data directory if it doesn't exist
os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/search_fixed.png", dpi=300, bbox_inches="tight")



Dynamic

In [ ]:
dynamic_terminal_points = dynamic_terminal_points_episodal[0]
#Seperate quasismooth and nonquasismooth terminal points for plotting
dynamic_terminal_points_qs = []
dynamic_terminal_points_nonqs = []
qs = pd.read_csv("../data/quasismooth/quasismooth_terminal.csv", header=None).values.astype(int).tolist()
qs = [tuple(point[1:]) for point in qs]
for point in dynamic_terminal_points:
    if tuple(point) in qs:
         dynamic_terminal_points_qs.append(tuple(point))
    else:
        dynamic_terminal_points_nonqs.append(tuple(point))

#Combine searched points from all episodes for plotting
dynamic_searched_points = []
for point in dynamic_searched_points_episodal[0]:
    dynamic_searched_points.append(tuple(point))

#Font settings
plt.rcParams.update({
    "text.usetex": False,       # don't require external LaTeX
    "font.family": "serif",     # generic family (will map automatically)
    "font.serif": ["DejaVu Serif"],  # built-in, always available
    "mathtext.fontset": "cm",   # use Computer Modern for math text
})

#Instantiate plot
fig, axs = plt.subplots(2, 3 , figsize=(30,20))
axs = axs.flatten()

#Project the data via (a1,...,an) -> (ai+1, ai+2) for i=0,...,n-2
for i in range(state_dim - 1):
    ax = axs[i]
    i1 = i
    i2 = i + 1

    #Determine grid size
    max_value = int(max(
        max(point[i1] for point in dynamic_searched_points),
        max(point[i2] for point in dynamic_searched_points),
    ))

    #Grid points
    X, Y = np.meshgrid(range(1, max_value + 1), range(1, max_value + 1))
    X = X.flatten()
    Y = Y.flatten()

    #Marker size
    ax_width = ax.get_window_extent().width
    ax_height = ax.get_window_extent().height
    marker_size = (min(ax_width/ max_value, ax_height/ max_value) * 0.85) ** 2 

    #Grey points where ai<=aj
    grey_mask = X <= Y
    ax.scatter(X[grey_mask], Y[grey_mask], color='#eeeee4', s=marker_size, alpha=1, label='Searchable')

    #Searched points
    ax_ay_searched = list(set([(point[i1],point[i2]) for point in dynamic_searched_points]))
    ax.scatter(*zip(*ax_ay_searched), color='orange', s=marker_size, alpha=1, label='Searched')

    #Quasismooth terminal points
    ax_ay_terminal = list(set([(point[i1],point[i2]) for point in dynamic_terminal_points_qs]))
    ax.scatter(*zip(*ax_ay_terminal), color='black', s=marker_size, alpha=1, label='Quasismooth terminal')

    #Nonquasismooth terminal points
    ax_ay_terminal = list(set([(point[i1],point[i2]) for point in dynamic_terminal_points_nonqs]))
    ax.scatter(*zip(*ax_ay_terminal), color='blue', s=marker_size, alpha=0.5, label='Nonquasismooth terminal')

    #Lablels and title
    ax.set_xlabel(f'$a_{i1+1}$', fontsize=40, color='grey')
    ax.set_ylabel(f'$a_{i2+1}$', fontsize=40, color='grey')

    #Format plot
    num_bins = 6
    ticks = list(np.linspace(0, max_value, num_bins))
    ticks.append(1)
    ticks = sorted(set(ticks))
    ticks = [int(t) for t in ticks if t != 0]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis='both', which='both', length=0, labelsize=20, colors="grey")

axs[state_dim - 1].axis('off')

#Legend
h, l = ax.get_legend_handles_labels()
axs[state_dim - 1].legend(
    h, l,
    frameon=False,
    markerscale=100,
    loc='center',
    fontsize=40,
    title=r'$\mathbf{Point\ Types}$',
    title_fontsize=45
)

plt.tight_layout()
plt.show()

#Create data directory if it doesn't exist
os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/search_dynamic.png", dpi=300, bbox_inches="tight")

Dynamic Search Hero 

In [ ]:
#Font settings
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",     
    "font.serif": ["DejaVu Serif"],  
    "mathtext.fontset": "cm",   
})

#Instantiate plot
fig, ax = plt.subplots(figsize=(20,20))

#Pick which coordinates to project onto (here we choose (a5, a6))
i = 4
i1 = i
i2 = i + 1

#Determine grid size
max_value = int(max(
    max(point[i1] for point in dynamic_searched_points),
    max(point[i2] for point in dynamic_searched_points)
))

#Grid points
X, Y = np.meshgrid(range(1, max_value + 1), range(1, max_value + 1))
X = X.flatten()
Y = Y.flatten()

#Dynamic marker size
ax_width = ax.get_window_extent().width
ax_height = ax.get_window_extent().height
marker_size = (min(ax_width/ max_value, ax_height/ max_value) * 0.85) ** 2 

#Grey points where ai<=aj
grey_mask = X <= Y
ax.scatter(X[grey_mask], Y[grey_mask], color='#eeeee4', s=marker_size, alpha=1, label='Searchable')

#Searched points
ax_ay_searched = list(set([(point[i1],point[i2]) for point in dynamic_searched_points]))
ax.scatter(*zip(*ax_ay_searched), color='orange', s=marker_size, alpha=1, label='Searched')

#Quasismooth terminal points
ax_ay_terminal = list(set([(point[i1],point[i2]) for point in dynamic_terminal_points_qs]))
ax.scatter(*zip(*ax_ay_terminal), color='black', s=marker_size, alpha=1, label='Quasismooth terminal (known)')

#Nonquasismooth terminal points
ax_ay_terminal = list(set([(point[i1],point[i2]) for point in dynamic_terminal_points_nonqs]))
ax.scatter(*zip(*ax_ay_terminal), color='blue', s=marker_size, alpha=0.5, label='Nonquasismooth terminal (new)')

#Lablels and title
ax.set_xlabel(f'$a_{i1+1}$', fontsize=40, color='grey')
ax.set_ylabel(f'$a_{i2+1}$', fontsize=40, color='grey')

#Format plot
num_bins = 6
ticks = list(np.linspace(0, max_value, num_bins))
ticks.append(1)
ticks = sorted(set(ticks))
ticks = [int(t) for t in ticks if t != 0] 
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis='both', which='both', length=0, labelsize=20, colors="grey")

ax.axis('off')

#Legend
h, l = ax.get_legend_handles_labels()
ax.legend(
    h, l,
    frameon=False,
    markerscale=50,
    loc=(0.35, 0.1),
    fontsize=40,
    title=r'$\mathbf{Points}$',
    title_fontsize=45,
)

plt.tight_layout()
plt.show()

#Create data directory if it doesn't exist
os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/search_dynamic_hero.png", dpi=300, bbox_inches="tight")



## $\S3$  Distances

Fixed Distances

In [ ]:
# Load qs (drop the first CSV column)
qs = pd.read_csv("../data/quasismooth/quasismooth_terminal.csv", header=None).values.astype(int).tolist()
qs = [tuple(point[1:]) for point in qs]
qs_set = set(qs)

# All fixed terminal points (unique); the non-qs ones are the queries
all_fixed = list(dict.fromkeys(tuple(p) for p in fixed_terminal_points))
fixed_terminal_points_nonqs = [p for p in all_fixed if p not in qs_set]
print(len(fixed_terminal_points_nonqs), "of", len(all_fixed), "fixed terminal points are not in qs.")

# Tree is built from all fixed terminal points; queries are the fixed terminal points not in qs
tree_list = all_fixed
tree_points = np.array(tree_list)
query_points = np.array(fixed_terminal_points_nonqs)

# L^1 metric (p=1); k=2 because the nearest hit is the point itself
tree = cKDTree(tree_points)
dists, idx = tree.query(query_points, k=2, p=1, workers=-1)

# Skip the point itself: if the closest hit is at distance 0, take the second one
col = (dists[:, 0] == 0).astype(int)
rows = np.arange(len(query_points))
nearest_idx = idx[rows, col]
nearest_dist = dists[rows, col]

# (point, closest other fixed terminal point, L^1 distance)
fixed_distances = [
    (tuple(q), tree_list[j], int(d))
    for q, j, d in zip(query_points.tolist(), nearest_idx, nearest_dist)
]

Store data

In [ ]:
fixed_distances_flat = [list(point1) + list(point2) + [dist] for point1, point2, dist in fixed_distances]
np.savetxt("../data/fixed/fixed_distances/fixed_distances.csv", fixed_distances_flat, delimiter=",")

Dynamic Distances

In [ ]:
# Load qs (drop the first CSV column)
qs = pd.read_csv("../data/quasismooth/quasismooth_terminal.csv", header=None).values.astype(int).tolist()
qs = [tuple(point[1:]) for point in qs]
qs_set = set(qs)

dynamic_distances_episodal = []  # one list of (point, closest_point, distance) per episode

for episode, episode_points in enumerate(dynamic_terminal_points_episodal, start=1):
    # All dynamic terminal points of this episode (unique); the non-qs ones are the queries
    all_dynamic = list(dict.fromkeys(tuple(int(w) for w in p) for p in episode_points))
    dynamic_terminal_points_nonqs = [p for p in all_dynamic if p not in qs_set]
    print(f"Episode {episode}: {len(dynamic_terminal_points_nonqs)} of {len(all_dynamic)} "
          f"dynamic terminal points are not in qs.")

    # Need at least one query point and two tree points (the point itself plus a neighbour)
    if len(dynamic_terminal_points_nonqs) == 0 or len(all_dynamic) < 2:
        dynamic_distances_episodal.append([])
        continue

    # Tree is built from all dynamic terminal points of the episode; queries are those not in qs
    tree_list = all_dynamic
    tree_points = np.array(tree_list)
    query_points = np.array(dynamic_terminal_points_nonqs)

    # L^1 metric (p=1); k=2 because the nearest hit is the point itself
    tree = cKDTree(tree_points)
    dists, idx = tree.query(query_points, k=2, p=1, workers=-1)

    # Skip the point itself: if the closest hit is at distance 0, take the second one
    col = (dists[:, 0] == 0).astype(int)
    rows = np.arange(len(query_points))
    nearest_idx = idx[rows, col]
    nearest_dist = dists[rows, col]

    # (point, closest other dynamic terminal point, L^1 distance)
    dynamic_distances = [
        (tuple(q), tree_list[j], int(d))
        for q, j, d in zip(query_points.tolist(), nearest_idx, nearest_dist)
    ]
    dynamic_distances_episodal.append(dynamic_distances)

Store Data

In [ ]:
os.makedirs("../data/dynamic/dynamic_distances", exist_ok=True)

for i, dynamic_distances in enumerate(dynamic_distances_episodal, start=1):
    # flatten data before storing to csv: point1 weights, point2 weights, distance
    dynamic_distances_flat = [list(point1) + list(point2) + [dist] for point1, point2, dist in dynamic_distances]
    np.savetxt(f"../data/dynamic/dynamic_distances/dynamic_distances_episode_{i}.csv", dynamic_distances_flat, delimiter=",", fmt="%d")

Ablation Distances

In [ ]:
# Load qs (drop the first CSV column)
qs = pd.read_csv("../data/quasismooth/quasismooth_terminal.csv", header=None).values.astype(int).tolist()
qs = [tuple(point[1:]) for point in qs]
qs_set = set(qs)

ablation_distances_episodal = []  # one list of (point, closest_point, distance) per episode

for episode, episode_points in enumerate(ablation_terminal_points_episodal, start=1):
    # All ablation terminal points of this episode (unique); the non-qs ones are the queries
    all_ablation = list(dict.fromkeys(tuple(int(w) for w in p) for p in episode_points))
    ablation_terminal_points_nonqs = [p for p in all_ablation if p not in qs_set]
    print(f"Episode {episode}: {len(ablation_terminal_points_nonqs)} of {len(all_ablation)} "
          f"ablation terminal points are not in qs.")

    # Need at least one query point and two tree points (the point itself plus a neighbour)
    if len(ablation_terminal_points_nonqs) == 0 or len(all_ablation) < 2:
        ablation_distances_episodal.append([])
        continue

    # Tree is built from all ablation terminal points of the episode; queries are those not in qs
    tree_list = all_ablation
    tree_points = np.array(tree_list)
    query_points = np.array(ablation_terminal_points_nonqs)

    # L^1 metric (p=1); k=2 because the nearest hit is the point itself
    tree = cKDTree(tree_points)
    dists, idx = tree.query(query_points, k=2, p=1, workers=-1)

    # Skip the point itself: if the closest hit is at distance 0, take the second one
    col = (dists[:, 0] == 0).astype(int)
    rows = np.arange(len(query_points))
    nearest_idx = idx[rows, col]
    nearest_dist = dists[rows, col]

    # (point, closest other ablation terminal point, L^1 distance)
    ablation_distances = [
        (tuple(q), tree_list[j], int(d))
        for q, j, d in zip(query_points.tolist(), nearest_idx, nearest_dist)
    ]
    ablation_distances_episodal.append(ablation_distances)

Store Data

In [ ]:
os.makedirs("../data/ablation/ablation_distances", exist_ok=True)

for i, ablation_distances in enumerate(ablation_distances_episodal, start=1):
    # flatten data before storing to csv: point1 weights, point2 weights, distance
    ablation_distances_flat = [list(point1) + list(point2) + [dist] for point1, point2, dist in ablation_distances]
    np.savetxt(f"../data/ablation/ablation_distances/ablation_distances_episode_{i}.csv", ablation_distances_flat, delimiter=",", fmt="%d")

Load Data

In [ ]:
#Dynamic Heuristic Search Data
dynamic_settings = json.load(open("../data/dynamic/dynamic_settings.json"))
max_episodes = dynamic_settings["max_episodes"]
state_dim = dynamic_settings["state_dim"]

#Fixed
fixed_distances_flat = pd.read_csv("../data/fixed/fixed_distances/fixed_distances.csv", header=None).values
#unflatten data
fixed_distance_data = []
for row in fixed_distances_flat:
    point1 = tuple(row[:state_dim])
    point2 = tuple(row[state_dim:2*state_dim])
    dist = row[-1]
    fixed_distance_data.append((point1, point2, dist))

#Dynamic
dynamic_distance_data_episodal = []   # (point1, point2, dist) tuples per episode
for i in range(1, max_episodes + 1):
    path = f"../data/dynamic/dynamic_distances/dynamic_distances_episode_{i}.csv"
    dynamic_distances = []
    if os.path.getsize(path) > 0:  # episodes with no distances are saved as empty files
        dynamic_distances_flat = pd.read_csv(path, header=None).values
        #unflatten data
        for row in dynamic_distances_flat:
            point1 = tuple(row[:state_dim])
            point2 = tuple(row[state_dim:2*state_dim])
            dist = row[-1]
            dynamic_distances.append((point1, point2, dist))
    dynamic_distance_data_episodal.append(dynamic_distances)

#Ablation
ablation_settings = json.load(open("../data/ablation/ablation_settings.json"))
max_episodes = ablation_settings["max_episodes"]
state_dim = ablation_settings["state_dim"]

ablation_distance_data_episodal = []  # (point1, point2, dist) tuples per episode
for i in range(1, max_episodes + 1):
    path = f"../data/ablation/ablation_distances/ablation_distances_episode_{i}.csv"
    ablation_distances = []
    if os.path.getsize(path) > 0:  # episodes with no distances are saved as empty files
        ablation_distances_flat = pd.read_csv(path, header=None).values
        #unflatten data
        for row in ablation_distances_flat:
            point1 = tuple(row[:state_dim])
            point2 = tuple(row[state_dim:2*state_dim])
            dist = row[-1]
            ablation_distances.append((point1, point2, dist))
    ablation_distance_data_episodal.append(ablation_distances)

Remove examples with $d\leq 200$

In [ ]:

# Distance Data Summary (d >= 0)
print("-------- Distance Data Summary with d >= 0 ----")
print(f"Fixed: {len(fixed_distance_data)} points")

dynamic_counts = [len(episode) for episode in dynamic_distance_data_episodal]
ablation_counts = [len(episode) for episode in ablation_distance_data_episodal]

for i, count in enumerate(dynamic_counts, start=1):
    print(f"Dynamic Episode {i}: {count} points")
print(f"Dynamic Mean ± Std: {np.mean(dynamic_counts):.1f} ± {np.std(dynamic_counts, ddof=1):.1f} points")

for i, count in enumerate(ablation_counts, start=1):
    print(f"Ablation Episode {i}: {count} points")
print(f"Ablation Mean ± Std: {np.mean(ablation_counts):.1f} ± {np.std(ablation_counts, ddof=1):.1f} points")


# Filter for degree >= 200
fixed_distance_data_200 = [(p1, p2, d) for p1, p2, d in fixed_distance_data if sum(p1) + alpha_X > 200]
dynamic_distance_data_episodal_200 = [[(p1, p2, d) for p1, p2, d in episode if sum(p1) + alpha_X > 200] for episode in dynamic_distance_data_episodal]
ablation_distance_data_episodal_200 = [[(p1, p2, d) for p1, p2, d in episode if sum(p1) + alpha_X > 200] for episode in ablation_distance_data_episodal]


# Distance Data Summary (d >= 200)
print("-------- Distance Data Summary with d > 200 ----")
print(f"Fixed: {len(fixed_distance_data_200)} points")

dynamic_counts_200 = [len(episode) for episode in dynamic_distance_data_episodal_200]
ablation_counts_200 = [len(episode) for episode in ablation_distance_data_episodal_200]

for i, count in enumerate(dynamic_counts_200, start=1):
    print(f"Dynamic Episode {i}: {count} points")
print(f"Dynamic Mean ± Std: {np.mean(dynamic_counts_200):.1f} ± {np.std(dynamic_counts_200, ddof=1):.1f} points")

for i, count in enumerate(ablation_counts_200, start=1):
    print(f"Ablation Episode {i}: {count} points")
print(f"Ablation Mean ± Std: {np.mean(ablation_counts_200):.1f} ± {np.std(ablation_counts_200, ddof=1):.1f} points")


High Degree, and Furthest Examples

In [ ]:

# Format points as tuples of Python integers
def format_point(point):
    return tuple(map(int, point))

# Fixed
print("--------------------FIXED SEARCH--------------------")

distance_fixed_max = max(dist for _, _, dist in fixed_distance_data_200)
hard_to_reach_fixed_points = len([data for data in fixed_distance_data_200 if data[2] >= 10])

print("Number of points hard to reach (distance >= 10):", hard_to_reach_fixed_points)

# Maximum degree (irrespective of distance)
p1_max, p2_max, dist_max = max(fixed_distance_data_200, key=lambda x: sum(x[0]) + alpha_X)
degree_fixed_max = int(sum(p1_max) + alpha_X)

print(f"\nMaximum degree (irrespective of distance): {degree_fixed_max}")
print("Point", format_point(p1_max), "from:", format_point(p2_max), "at distance:", int(dist_max))

# Furthest points
print("\nFurthest points (up to 5 examples, ordered by decreasing degree):")

furthest_fixed = [(p1, p2, dist) for p1, p2, dist in fixed_distance_data_200 if dist == distance_fixed_max]
furthest_fixed.sort(key=lambda x: sum(x[0]) + alpha_X, reverse=True)

print(f"Maximum distance: {int(distance_fixed_max)}")
print(f"Total points attaining maximum distance: {len(furthest_fixed)}")

for p1, p2, dist in furthest_fixed[:5]:
    degree = int(sum(p1) + alpha_X)
    print("Point", format_point(p1), "degree:", degree, "from:", format_point(p2), "at distance:", int(dist))

# Dynamic
print("--------------------DYNAMIC SEARCH--------------------")

for i, episode in enumerate(dynamic_distance_data_episodal_200):
    print(f"\nRun {i+1}")

    if not episode:
        print("No points found.")
        continue

    distance_dynamic_max = max(dist for _, _, dist in episode)
    hard_to_reach_dynamic_points = len([data for data in episode if data[2] >= 10])

    print("Number of points hard to reach (distance >= 10):", hard_to_reach_dynamic_points)

    # Maximum degree (irrespective of distance)
    p1_max, p2_max, dist_max = max(episode, key=lambda x: sum(x[0]) + alpha_X)
    degree_dynamic_max = int(sum(p1_max) + alpha_X)

    print(f"Maximum degree (irrespective of distance): {degree_dynamic_max}")
    print("Point", format_point(p1_max), "from:", format_point(p2_max), "at distance:", int(dist_max))

    # Furthest points
    print(f"Maximum distance: {int(distance_dynamic_max)}")

    for p1, p2, dist in episode:
        if dist == distance_dynamic_max:
            degree = int(sum(p1) + alpha_X)
            print("Furthest point", format_point(p1), "degree:", degree,"from:", format_point(p2), "at distance:", int(dist))

# Ablation
print("--------------------ABLATION SEARCH--------------------")

for i, episode in enumerate(ablation_distance_data_episodal_200):
    print(f"\nRun {i+1}")

    if not episode:
        print("No points found.")
        continue

    distance_ablation_max = max(dist for _, _, dist in episode)
    hard_to_reach_ablation_points = len([data for data in episode if data[2] >= 10])

    print("Number of points hard to reach (distance >= 10):", hard_to_reach_ablation_points)

    # Maximum degree (irrespective of distance)
    p1_max, p2_max, dist_max = max(episode, key=lambda x: sum(x[0]) + alpha_X)
    degree_ablation_max = int(sum(p1_max) + alpha_X)

    print(f"Maximum degree (irrespective of distance): {degree_ablation_max}")
    print("Point", format_point(p1_max),"from:", format_point(p2_max),"at distance:", int(dist_max))

    # Furthest points
    print(f"Maximum distance: {int(distance_ablation_max)}")

    for p1, p2, dist in episode:
        if dist == distance_ablation_max:
            degree = int(sum(p1) + alpha_X)
            print("Furthest point", format_point(p1),"degree:", degree,"from:", format_point(p2),"at distance:", int(dist))


Fixed and Dynamic Histograms

In [ ]:

# Extract distances
fixed_distances_200 = [dist for _, _, dist in fixed_distance_data_200]
dynamic_distances_episodal_200 = [[dist for _, _, dist in episode] for episode in dynamic_distance_data_episodal_200]

# Font settings
plt.rcParams.update({"text.usetex": False, "font.family": "serif", "font.serif": ["DejaVu Serif"], "mathtext.fontset": "cm"})

# --- 2 rows, 3 columns ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
fig.set_constrained_layout_pads(hspace=0.17, h_pad=0.08)
axes = axes.flatten()

all_data = [fixed_distances_200, *dynamic_distances_episodal_200]
titles = [r"$\mathcal{F}$", *[rf"$\mathcal{{D}}_{{{i}}}$" for i in range(1, 6)]]
xlabels = [r"$D_{\mathcal{F}}(p)$", *[rf"$D_{{\mathcal{{D}}_{{{i}}}}}(p)$" for i in range(1, 6)]]

for i, (ax, data, title, xlabel) in enumerate(zip(axes, all_data, titles, xlabels)):

    local_min = int(min(data))
    local_max = int(max(data))
    bins = np.arange(local_min, local_max + 2) - 0.5

    counts, _, patches = ax.hist(data, bins=bins, color="blue", alpha=0.5, rwidth=0.85)

    # Axes formatting
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)

    ax.spines["bottom"].set_color("grey")

    # Show EVERY integer distance on the x-axis
    ax.set_xticks(np.arange(local_min, local_max + 1))

    ax.tick_params(axis="x", colors="grey", labelsize=11, length=0, pad=5, rotation=45)
    ax.tick_params(axis="y", colors="grey", labelsize=12, length=0, pad=5)

    # Align rotated x-axis labels
    plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

    ax.set_xlim(local_min - 0.7, local_max + 0.7)

    # Extra vertical space for frequency annotations
    ax.set_ylim(0, max(counts) * (1.18 if i == 0 else 1.42))

    ax.set_title(title, fontsize=20, color="black", pad=12)
    ax.set_xlabel(xlabel, fontsize=17, color="grey", labelpad=8)

    if i % 3 == 0:
        ax.set_ylabel("Frequency", fontsize=15, color="grey", labelpad=8)

    # Annotate EVERY nonzero bar
    for patch, count in zip(patches, counts):
        if count == 0:
            continue

        x = patch.get_x() + patch.get_width() / 2
        y = patch.get_height()

        ax.annotate(
            f"{int(count):,}",
            xy=(x, y),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            rotation=0 if i == 0 else 90,
            fontsize=11,
            color="grey",
            clip_on=False
        )

# Save before displaying
os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/fixed_dynamic_histogram.png", dpi=300, bbox_inches="tight")

plt.show()


Ablation Histogram

In [ ]:
# Extract ablation distances
ablation_distances_episodal_200 = [[dist for _, _, dist in episode] for episode in ablation_distance_data_episodal_200]

# Font settings
plt.rcParams.update({"text.usetex": False, "font.family": "serif", "font.serif": ["DejaVu Serif"], "mathtext.fontset": "cm"})

# --- 2 rows, 3 columns ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
fig.set_constrained_layout_pads(hspace=0.17, h_pad=0.08)
axes = axes.flatten()

# Hide unused sixth panel
axes[5].axis("off")

all_data = ablation_distances_episodal_200
titles = [rf"$\mathcal{{A}}_{{{i}}}$" for i in range(1, 6)]
xlabels = [rf"$D_{{\mathcal{{A}}_{{{i}}}}}(p)$" for i in range(1, 6)]

for i, (ax, data, title, xlabel) in enumerate(zip(axes[:5], all_data, titles, xlabels)):

    local_min = int(min(data))
    local_max = int(max(data))
    bins = np.arange(local_min, local_max + 2) - 0.5

    counts, _, patches = ax.hist(data, bins=bins, color="blue", alpha=0.5, rwidth=0.85)

    # Axes formatting
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)

    ax.spines["bottom"].set_color("grey")

    # Show EVERY integer distance on the x-axis
    ax.set_xticks(np.arange(local_min, local_max + 1))

    ax.tick_params(axis="x", colors="grey", labelsize=11, length=0, pad=5, rotation=45)
    ax.tick_params(axis="y", colors="grey", labelsize=12, length=0, pad=5)

    # Align rotated x-axis labels
    plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

    ax.set_xlim(local_min - 0.7, local_max + 0.7)

    # Extra vertical space for annotations
    ax.set_ylim(0, max(counts) * 1.42)

    ax.set_title(title, fontsize=20, color="black", pad=12)
    ax.set_xlabel(xlabel, fontsize=17, color="grey", labelpad=8)

    if i % 3 == 0:
        ax.set_ylabel("Frequency", fontsize=15, color="grey", labelpad=8)

    # Annotate EVERY nonzero bar vertically
    for patch, count in zip(patches, counts):
        if count == 0:
            continue

        x = patch.get_x() + patch.get_width() / 2
        y = patch.get_height()

        ax.annotate(
            f"{int(count):,}",
            xy=(x, y),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=11,
            color="grey",
            clip_on=False
        )

# Save before displaying
os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/ablation_histogram.png", dpi=300, bbox_inches="tight")

plt.show()
